# Getting the Data ready for the models including the Numpy

## First we need to get the data cleaned

* only adding the labels (supervised)
* removing the ligand row
* replace empty rows with 0



In [4]:
import pandas as pd
import os

# Define input config for each drug
drug_files = {
    "AMP": {
        "path": "Data/New/fp_comp_dataframe_AMP.csv",
        "drug_name": "AMP",
        "bond_type": "inward"
    },

    # "BZT": {
    #     "path": "Data/New/fp_comp_dataframe_BZT.csv",
    #     "drug_name": "BZT",
    #     "bond_type": "inward"
    # },

    # "COC": {
    #     "path": "Data/New/fp_comp_dataframe_COC.csv",
    #     "drug_name": "COC",
    #     "bond_type": "outward"
    # },

    # "DOP": {
    #     "path": "Data/New/fp_comp_dataframe_DOP.csv",
    #     "drug_name": "DOP",
    #     "bond_type": "occluded"

    # },
    "MAPB":{
        "path": "Data/New/fp_comp_dataframe_MAPB.csv",
        "drug_name": "MAPB",
        "bond_type": "inward"
    },

    "MPH":{
        "path": "Data/New/fp_comp_dataframe_MPH.csv",
        "drug_name": "MPH",
        "bond_type": "outward"
    },

    "MDMA":{
        "path": "Data/Original/fp_comp_dataframe_MDMA.csv",
        "drug_name": "MDMA",
        "bond_type": "inward"
    }
}

# Processing loop
for drug, config in drug_files.items():
    # Load original CSV
    df = pd.read_csv(config["path"])

    # STEP 1: Drop the third row (index 2), which contains mostly NaN values
    df_cleaned = df.drop(index=2).reset_index(drop=True)

    # STEP 2: Extract header rows
    header_residues = df_cleaned.iloc[0]
    header_types = df_cleaned.iloc[1]

    # STEP 3: Create MultiIndex
    multi_index = pd.MultiIndex.from_arrays([header_residues, header_types])

    # STEP 4: Drop header rows
    df_cleaned = df_cleaned.drop(index=[0, 1]).reset_index(drop=True)

    # STEP 5: Apply MultiIndex
    df_cleaned.columns = multi_index

    # STEP 6: Insert correct frame column
    frame_col = df.iloc[3:, 0].reset_index(drop=True)
    df_cleaned.insert(0, ("meta", "frame"), frame_col.astype(int))

    # STEP 6.5: Drop redundant ("protein", "interaction") column if present
    if ("protein", "interaction") in df_cleaned.columns:
        df_cleaned = df_cleaned.drop(columns=[("protein", "interaction")])

    # STEP 7: Convert numerics
    df_cleaned = df_cleaned.apply(pd.to_numeric, errors='ignore')

    # STEP 8: Fill NaN with 0
    df_cleaned = df_cleaned.fillna(0)

    # STEP 9: Add meta columns
    df_cleaned[("meta", "bond_type")] = config["bond_type"]
    df_cleaned[("meta", "drug_name")] = config["drug_name"]

    # STEP 10: Reorder columns
    df_cleaned = df_cleaned[
        [col for col in df_cleaned.columns if col[0] != "meta" or col[1] == "frame"] +
        [("meta", "bond_type"), ("meta", "drug_name")]
    ]

    # Save cleaned version
    output_path = f"Data/New/processed/cleaned_{config['drug_name']}.csv"            #out directory
    df_cleaned.to_csv(output_path, index=False)
    print(f"Saved cleaned CSV for {drug}: {output_path}")


Saved cleaned CSV for AMP: Data/New/processed/cleaned_AMP.csv
Saved cleaned CSV for MAPB: Data/New/processed/cleaned_MAPB.csv
Saved cleaned CSV for MPH: Data/New/processed/cleaned_MPH.csv
Saved cleaned CSV for MDMA: Data/New/processed/cleaned_MDMA.csv


C:\Users\amirt\AppData\Local\Temp\ipykernel_16576\1818585725.py:79: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_cleaned = df_cleaned.apply(pd.to_numeric, errors='ignore')
C:\Users\amirt\AppData\Local\Temp\ipykernel_16576\1818585725.py:79: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_cleaned = df_cleaned.apply(pd.to_numeric, errors='ignore')
C:\Users\amirt\AppData\Local\Temp\ipykernel_16576\1818585725.py:79: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_cleaned = df_cleaned.apply(pd.to_numeric, errors='ignore')
C:\Users\amirt\AppData\Local\Temp\ipykernel_16576\1818585725.py:79: FutureWarning: errors='ignore' is deprecated and will

## changing to Numpy format 

we will be obtaining the following information:

* X_all: a 2D numpy that captures frames and the features such as: (n_samples, n_features)
* Y_all: 1D array that encodes the labels e.g., 0, 1, 2 AKA 'inward' 'occluded' 'outward'
* feature_names: captures all feature column names e.g., "ALA480.P.VdWContact"

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Set paths
input_dir = "Data/New/trained"
output_dir = "models/v3"
os.makedirs(output_dir, exist_ok=True)

# Step 1: Get full feature column set
all_columns = set()
meta_cols = [("meta", "frame"), ("meta", "bond_type"), ("meta", "drug_name")]

for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(input_dir, file), header=[0, 1])
        feature_cols = [col for col in df.columns if col[0] != "meta"]
        all_columns.update(feature_cols)

# Sort and freeze final feature order
all_columns = sorted(all_columns)
final_columns = all_columns  # Only features

# Step 2: Process all files
X_list, y_list, drug_names = [], [], []

for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(input_dir, file), header=[0, 1])
        drug = df[("meta", "drug_name")].values[0] if ("meta", "drug_name") in df.columns else "unknown"
        y = df[("meta", "bond_type")].values

        # Align features to global set, fill missing with 0
        df_features = df.reindex(columns=final_columns, fill_value=0)
        X = df_features.to_numpy(dtype=np.float32)

        X_list.append(X)
        y_list.append(y)
        drug_names.extend([drug] * len(y))

# Stack
X_all = np.vstack(X_list)
y_all_series = np.concatenate(y_list)

# Encode labels
le = LabelEncoder()
y_all = le.fit_transform(y_all_series)
y_labels = le.classes_

# Show class counts
print("Original class distribution:", Counter(y_all))

# Undersample
rus = RandomUnderSampler(random_state=42)
X_balanced, y_balanced = rus.fit_resample(X_all, y_all)
print("Balanced class distribution:", Counter(y_balanced))

# Save
np.save(os.path.join(output_dir, "X_all.npy"), X_balanced)
np.save(os.path.join(output_dir, "y_all.npy"), y_balanced)
np.save(os.path.join(output_dir, "y_labels.npy"), y_labels)
np.save(os.path.join(output_dir, "feature_names.npy"), np.array(final_columns, dtype=object))

print("✅ Saved aligned & balanced dataset to:", output_dir)


Original class distribution: Counter({np.int64(1): 3287, np.int64(0): 3083, np.int64(2): 2879})
Balanced class distribution: Counter({np.int64(0): 2879, np.int64(1): 2879, np.int64(2): 2879})
✅ Saved aligned & balanced dataset to: models/v3


In [6]:
# Shape check 
output_dir = "models/v3"

# Load arrays
X_all = np.load(os.path.join(output_dir, "X_all.npy"))
y_all = np.load(os.path.join(output_dir, "y_all.npy"))
y_labels = np.load(os.path.join(output_dir, "y_labels.npy"), allow_pickle=True)
feature_names = np.load(os.path.join(output_dir, "feature_names.npy"), allow_pickle=True)

# Print shapes
print("✅ Shapes of loaded arrays:")
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print("y_labels:", y_labels.shape)
print("feature_names:", feature_names.shape)


✅ Shapes of loaded arrays:
X_all: (8637, 87)
y_all: (8637,)
y_labels: (3,)
feature_names: (87, 2)
